# Day 017 — Exercise 4: Tool Turn

**Goal:** Implement `tool_turn(history, user_input, tools, registry, model)` — the complete tool-use loop that handles both direct replies and tool calls.

In [ ]:
import ollama

In [ ]:
import ast, operator

def calculate(expression: str) -> str:
    """Evaluate a safe arithmetic expression and return the result as a string."""
    allowed = {
        ast.Add: operator.add, ast.Sub: operator.sub,
        ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg,
    }
    def _eval(node):
        if isinstance(node, ast.Constant):
            return node.value
        if isinstance(node, ast.BinOp):
            return allowed[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp):
            return allowed[type(node.op)](_eval(node.operand))
        raise ValueError(f"Unsafe expression: {ast.dump(node)}")
    result = _eval(ast.parse(expression, mode='eval').body)
    return str(result)

def get_weather(city: str) -> str:
    """Return a simulated current temperature for a city."""
    temperatures = {"london": "12\u00b0C", "tokyo": "22\u00b0C", "paris": "15\u00b0C",
                    "new york": "18\u00b0C", "sydney": "24\u00b0C"}
    temp = temperatures.get(city.lower(), "20\u00b0C")
    return f"The current temperature in {city} is {temp}."

TOOL_REGISTRY = {
    "calculate": calculate,
    "get_weather": get_weather,
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": (
                "Evaluate a mathematical expression and return the numeric result. "
                "Use this for any arithmetic including +, -, *, /, **, and %. "
                "Pass the expression as a string, e.g. '2847 * 193'."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A Python math expression, e.g. '12 * 34'",
                    }
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": (
                "Return the current temperature for a city. "
                "Use this when the user asks about current weather or temperature."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name, e.g. 'London' or 'Tokyo'",
                    }
                },
                "required": ["city"],
            },
        },
    },
]

In [ ]:
def has_tool_call(response: dict) -> bool:
    """Return True if the model's response contains at least one tool call."""
    return bool(response["message"].get("tool_calls"))

def extract_tool_call(response: dict) -> tuple[str, dict]:
    """Return (fn_name, fn_args) from the first tool call in the response."""
    call = response["message"]["tool_calls"][0]
    return call["function"]["name"], call["function"]["arguments"]

In [ ]:
def execute_tool(fn_name: str, fn_args: dict, registry: dict) -> str:
    """Look up fn_name in registry and call it with **fn_args."""
    if fn_name not in registry:
        raise KeyError(f"Unknown tool: {fn_name!r}")
    return registry[fn_name](**fn_args)

In [ ]:
def append_tool_result(
    messages: list[dict],
    assistant_msg: dict,
    tool_output: str,
) -> list[dict]:
    """Return new messages list with assistant tool-call msg and tool result appended."""
    tool_result = {"role": "tool", "content": tool_output}
    return messages + [assistant_msg, tool_result]

In [ ]:
def append_turn(history: list[dict], user_text: str, assistant_text: str) -> list[dict]:
    """Return a new history list with one user+assistant turn appended."""
    return history + [
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": assistant_text},
    ]

## Your Implementation

In [ ]:
def tool_turn(
    history: list[dict],
    user_input: str,
    tools: list[dict],
    registry: dict,
    model: str = "llama3.2",
) -> tuple[str, list[dict]]:
    """
    Send user_input to the model and handle any tool call it makes.

    Path A (direct reply): model replies → return (reply, append_turn(history, ...))
    Path B (tool call): detect → execute → append_tool_result → second model call
                        → return (reply, append_turn(history, ...))
    """
    # TODO: build messages = history + [user message dict]
    # TODO: call ollama.chat(model, messages, tools=tools)
    # TODO: if has_tool_call(response): execute tool, thread result, call again
    # TODO: extract reply = response["message"]["content"]
    # TODO: return (reply, append_turn(history, user_input, reply))
    pass

## Check Your Work

In [ ]:
import io, sys

def _run_checks():
    total = 5
    passed = 0
    calc_result = None

    # Check 1: tool_turn is defined
    try:
        assert 'tool_turn' in globals()
        passed += 1; print("\u2705 Check 1: tool_turn is defined")
    except Exception as e:
        print(f"\u274c Check 1: not defined \u2014 {e}")

    # Check 2: tool_turn returns (str, list)
    try:
        old = sys.stdout; sys.stdout = io.StringIO()
        calc_result = tool_turn([], "What is 12 * 34?", TOOLS, TOOL_REGISTRY)
        sys.stdout = old
        assert isinstance(calc_result, tuple) and len(calc_result) == 2, \
            f"expected tuple of length 2, got {type(calc_result)}"
        passed += 1; print("\u2705 Check 2: tool_turn returns a tuple of length 2")
    except Exception as e:
        sys.stdout = old
        print(f"\u274c Check 2: return type \u2014 {e}")

    # Check 3: reply contains '408' (correct result of 12 * 34)
    try:
        assert isinstance(calc_result, tuple), "Check 2 must pass first"
        reply, new_history = calc_result
        assert isinstance(reply, str) and "408" in reply, \
            f"expected '408' in reply, got {reply!r}"
        passed += 1; print("\u2705 Check 3: reply contains the correct calculation result")
    except Exception as e:
        print(f"\u274c Check 3: calculation result \u2014 {e}")

    # Check 4: new history has 2 messages (user+assistant)
    try:
        assert isinstance(calc_result, tuple), "Check 2 must pass first"
        _, new_history = calc_result
        assert isinstance(new_history, list) and len(new_history) == 2, \
            f"expected 2 messages (user+assistant), got {len(new_history) if isinstance(new_history, list) else type(new_history)}"
        passed += 1; print("\u2705 Check 4: new history has correct length")
    except Exception as e:
        print(f"\u274c Check 4: history length \u2014 {e}")

    # Check 5: tool_turn also works for a direct reply (no tool call)
    try:
        old = sys.stdout; sys.stdout = io.StringIO()
        direct_result = tool_turn([], "What is the capital of France?", TOOLS, TOOL_REGISTRY)
        sys.stdout = old
        reply2, hist2 = direct_result
        assert isinstance(reply2, str) and len(reply2.strip()) > 0
        assert isinstance(hist2, list) and len(hist2) == 2
        passed += 1; print("\u2705 Check 5: tool_turn works for direct replies too")
    except Exception as e:
        sys.stdout = old
        print(f"\u274c Check 5: direct reply path \u2014 {e}")

    if passed == total:
        print("\U0001f389 Exercise complete!")
    print(f"\nScore: {passed}/{total}")

_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
import ollama

def tool_turn(
    history: list[dict],
    user_input: str,
    tools: list[dict],
    registry: dict,
    model: str = "llama3.2",
) -> tuple[str, list[dict]]:
    messages = history + [{"role": "user", "content": user_input}]
    response = ollama.chat(model=model, messages=messages, tools=tools)

    if has_tool_call(response):
        fn_name, fn_args = extract_tool_call(response)
        tool_output = execute_tool(fn_name, fn_args, registry)
        messages = append_tool_result(messages, response["message"], tool_output)
        response = ollama.chat(model=model, messages=messages, tools=tools)

    reply = response["message"]["content"]
    return reply, append_turn(history, user_input, reply)
```

</details>